In [1]:
import pandas as pd

# Read both CSV files
darko_df = pd.read_csv('2025_draft_predictions.csv')
tankathon_df = pd.read_csv('tankathon_2025_draft.csv')

# Get only the 'name' column and columns that start with 'predict' from darko
prediction_cols = ['name'] + [col for col in darko_df.columns if col.startswith('predict')]
darko_subset = darko_df[prediction_cols]

# Perform left join: keep all rows from tankathon, add matching darko predictions
merged_df = tankathon_df.merge(darko_subset, on='name', how='left')

# Select only the desired columns
columns_to_keep = [
    'name', 'pre-draft_team', 'draft_position', 'position', 'draft_age',
    'PTS', 'REB', 'AST', 'BLK', 'STL', 'FG%', '3P%', 'predicted_peak_darko'
]
final_df = merged_df[columns_to_keep].copy()

# Clean up draft_position: extract number and rename
# Convert "1st" -> 1, "2nd" -> 2, etc.
final_df['draft_position'] = final_df['draft_position'].str.extract(r'(\d+)')[0].astype('Int64')
final_df.rename(columns={'draft_position': 'Actual Draft Position'}, inplace=True)

# Convert FG% and 3P% to percentages (multiply by 100)
final_df['FG%'] = final_df['FG%'] * 100
final_df['3P%'] = final_df['3P%'] * 100

# Round predicted_peak_darko to 2 decimal places
final_df['predicted_peak_darko'] = final_df['predicted_peak_darko'].round(2)

# Create Model Position: rank by predicted_peak_darko (highest to lowest)
# Use rank with method='min' to handle ties, and ascending=False for high to low
final_df['Model Position'] = final_df['predicted_peak_darko'].rank(ascending=False, method='min').astype('Int64')

# Create Difference column: Actual Draft Position - Model Position
# Positive means drafted higher than model predicted (overvalued)
# Negative means drafted lower than model predicted (undervalued)
final_df['Difference'] = final_df['Actual Draft Position'] - final_df['Model Position']

# Format the Difference column: add '+' for positive, '-' for zero, keep negative as-is
def format_difference(x):
    if pd.isna(x):
        return x
    elif x == 0:
        return '-'
    elif x > 0:
        return f'+{int(x)}'
    else:
        return str(int(x))

final_df['Difference'] = final_df['Difference'].apply(format_difference)

# Sort by Model Position for easier viewing
final_df = final_df.sort_values('Model Position')

# Save the final dataset
final_df.to_csv('2025_draft_final.csv', index=False)

print(f"Final dataset created!")
print(f"Total players: {len(final_df)}")
print(f"Players with predictions: {final_df['predicted_peak_darko'].notna().sum()}")
print(f"\nTop 10 by Model Position:")
print(final_df[['Model Position', 'name', 'Actual Draft Position', 'Difference', 'predicted_peak_darko']].head(10))
print(f"\nColumns in final dataset:")
print(final_df.columns.tolist())

Final dataset created!
Total players: 83
Players with predictions: 83

Top 10 by Model Position:
    Model Position                  name  Actual Draft Position Difference  \
31               1          Cooper Flagg                      1          -   
71               2          Kon Knueppel                      4         +2   
40               3  Collin Murray-Boyles                      9         +6   
33               4        V.J. Edgecombe                      3         -1   
2                5        Khaman Maluach                     10         +5   
14               6         Thomas Sorber                     15         +9   
16               7          Dylan Harper                      2         -5   
20               8           Adou Thiero                     36        +28   
55               9       Jase Richardson                     25        +16   
45              10        Chucky Hepburn                   <NA>        NaN   

    predicted_peak_darko  
31               

In [3]:
final_df.head()

,name,pre-draft_team,Actual Draft Position,position,draft_age,PTS,REB,AST,BLK,STL,FG%,3P%,predicted_peak_darko,Model Position,Difference
31,Cooper Flagg,Duke,1,SF/PF,18.50 yrs,19.2,7.5,4.2,1.4,1.4,48.1,38.5,2.29,1,-
71,Kon Knueppel,Duke,4,SG/SF,19.88 yrs,14.4,4.0,2.7,0.2,1.0,47.9,40.6,1.16,2,+2
40,Collin Murray-Boyles,South Carolina,9,PF,20.03 yrs,16.8,8.3,2.4,1.3,1.5,58.6,26.5,1.09,3,+6
33,V.J. Edgecombe,Baylor,3,SG,19.89 yrs,15.0,5.6,3.2,0.6,2.1,43.6,34.0,0.99,4,-1
2,Khaman Maluach,Duke,10,C,18.76 yrs,8.6,6.6,0.5,1.3,0.2,71.2,25.0,0.87,5,+5
